In [36]:
import torch
import torch.nn.functional as F
from transformer_lens import HookedTransformer
from tuned_lens import TunedLens
from transformers import AutoTokenizer

In [37]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

In [38]:
model_name = "EleutherAI/pythia-14m"
model: HookedTransformer = HookedTransformer.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loaded pretrained model EleutherAI/pythia-14m into HookedTransformer
Moving model to device:  mps


In [39]:
prompt = "the sky is blue and the grass is "
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    logits, cache = model.run_with_cache(input_ids, return_type="logits")
    hidden_states = [cache["hook_embed"]]  # start with embedding
    # Add transformer post-residuals
    for layer in range(model.cfg.n_layers):
        hidden_states.append(cache[f"blocks.{layer}.hook_resid_post"])

In [40]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleTunedLens(nn.Module):
    def __init__(self, d_model, vocab_size):
        super().__init__()
        # one linear layer per transformer layer (plus embedding)
        self.layers = nn.ModuleList([nn.Linear(d_model, vocab_size) for _ in range(7)]) # 1 embedding + 6 layers

    def forward(self, hidden_states):
        # hidden_states is a list: [embedding, layer0, layer1, ...]
        logits_per_layer = []
        for i, h in enumerate(hidden_states):
            # h: [batch, seq_len, d_model]
            logits = self.layers[i](h)
            logits_per_layer.append(logits)
        return logits_per_layer

In [41]:
vocab_size = model.cfg.d_vocab
d_model = model.cfg.d_model
n_layers = model.cfg.n_layers + 1  # embedding + all transformer layers

tuned_lens = SimpleTunedLens(d_model=d_model, vocab_size=vocab_size).to(device)

In [42]:
optimizer = torch.optim.Adam(tuned_lens.parameters(), lr=1e-3)
tuned_lens.train()

SimpleTunedLens(
  (layers): ModuleList(
    (0-6): 7 x Linear(in_features=128, out_features=50304, bias=True)
  )
)

In [43]:
targets = input_ids
loss = 0
for h in hidden_states:
    pred = tuned_lens([h])[0]  # [batch, seq_len, vocab_size]
    seq_len = pred.shape[1]

    # shift for next-token prediction
    pred_shift = pred[:, :-1, :]
    targets_shift = targets[:, 1:seq_len]

    pred_flat = pred_shift.reshape(-1, vocab_size)
    targets_flat = targets_shift.reshape(-1)

    loss += F.cross_entropy(pred_flat, targets_flat)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print(f" Single-step Tuned Lens training done. Loss: {loss.item():.4f}")


 Single-step Tuned Lens training done. Loss: 76.5620


In [44]:
top_k = 5
tuned_lens.eval()
print("\n=== Tuned Lens Predictions per Layer ===")

for layer_idx, h in enumerate(hidden_states):
    with torch.no_grad():
        logits_layer = tuned_lens([h])[0]  # [batch, seq_len, vocab_size]
        last_token_logits = logits_layer[0, -1]
        top_values, top_indices = torch.topk(last_token_logits, top_k)
        top_tokens = tokenizer.convert_ids_to_tokens(top_indices.cpu())

        print(f"\nLayer {layer_idx} (Tuned Lens):")
        for t, v in zip(top_tokens, top_values):
            print(f"  {t} : {v.item():.4f}")


=== Tuned Lens Predictions per Layer ===

Layer 0 (Tuned Lens):
  qr : 0.1386
  :{\ : 0.1362
  safe : 0.1353
  ĠRSA : 0.1315
  FY : 0.1305

Layer 1 (Tuned Lens):
  ĠRSA : 0.2329
  ĠMLB : 0.2286
  Î¹Ïĥ : 0.2255
  ĠPut : 0.2167
  rimination : 0.2137

Layer 2 (Tuned Lens):
  ention : 0.2296
  ĠWei : 0.2157
  Ġrelease : 0.2136
  Ġintox : 0.2108
  Ġ$-\ : 0.2101

Layer 3 (Tuned Lens):
  ZH : 0.3257
  cancers : 0.3102
  Ġpercol : 0.2917
  Ġstatue : 0.2880
  ĠPRE : 0.2867

Layer 4 (Tuned Lens):
  ĠNeed : 0.3711
  Ġundoubtedly : 0.3602
  opin : 0.3583
  tingham : 0.3540
  Ġhinge : 0.3414

Layer 5 (Tuned Lens):
  lan : 0.4989
  Swed : 0.4949
  andum : 0.4569
  Ġ&& : 0.4443
  Ġleg : 0.4297

Layer 6 (Tuned Lens):
  Ġquin : 0.5887
  lan : 0.5087
  tid : 0.4970
  Ġstoring : 0.4849
  Ġessa : 0.4734
